## Ejercicios
1. Lee los CSVs customers.csv, orders.csv, products.csv y regions.csv en DataFrames de Spark.

2. Muestra todos los clientes y su pedido más caro (ten en cuenta que un cliente puede tener varios pedidos registrados).

3. Muestra todos los clientes que nunca han hecho un pedido. Compruébalo obteniendo todos los clientes con pedidos y verificando que no estén en la tabla.

4. Encuentra los productos comprados por los 10 clientes que más dinero han gastado en total (ten en cuenta que un cliente puede haber comprado varios productos distintos).

In [ ]:
# ============================
# 1. Leer los CSVs en DataFrames
# ============================
customersDF = spark.read.option("header", True).option("inferSchema", True).csv("customers.csv")
ordersDF = spark.read.option("header", True).option("inferSchema", True).csv("orders.csv")
productsDF = spark.read.option("header", True).option("inferSchema", True).csv("products.csv")
regionsDF = spark.read.option("header", True).option("inferSchema", True).csv("regions.csv")


# ============================
# 2. Mostrar todos los clientes y su pedido más caro
# ============================
maxOrdersDF = ordersDF.groupBy("customer_id").agg(max("order_amount").alias("max_order_amount"))
customersDF.join(maxOrdersDF, "customer_id", "left").show()


# ============================
# 3. Mostrar todos los clientes que nunca han hecho un pedido
# ============================
customersWithoutOrdersDF = customersDF.join(ordersDF, customersDF.customer_id == ordersDF.customer_id, "left_anti")
customersWithoutOrdersDF.show()


# ============================
# 4. Encontrar los productos comprados por los 10 clientes que más gastaron
# ============================
# 4.1 Gastos totales por cliente
totalSpentDF = ordersDF.groupBy("customer_id").agg(sum("order_amount").alias("total_spent"))


# 4.2 Top 10 clientes que más gastaron
top10 = totalSpentDF.orderBy(totalSpentDF.total_spent.desc()).limit(10)


# 4.3 Pedidos de esos clientes
top10Orders = top10.join(ordersDF, "customer_id", "inner")


# 4.4 Productos comprados por esos clientes
top10Products = top10Orders.join(productsDF, "product_id", "inner").select("customer_id", "product_id", "product_name", "category", "price")
top10Products.show()